# Raw Docling + Docling-Graph Walkthrough

This notebook drops down to the **raw upstream libraries** —
`docling` and `docling_graph` — and walks a document through the full
extraction arc without our application's wrappers, Celery tasks, or
FastAPI services in the loop.

It mirrors, line-for-line where possible, what our two services
actually do:

| Step in this notebook | What it reproduces from our repo |
|---|---|
| Docling conversion | `docker/docling/app/converter.py` |
| Docling JSON export | `docker/docling/app/converter.py:242` |
| PipelineConfig build | `docker/docling-graph/app/config_builder.py:109-179` |
| Pass template resolution | `docker/docling-graph/app/bundles.py` |
| `run_pipeline(config)` | `docker/docling-graph/app/main.py:428` |
| Provenance build | `docker/docling-graph/app/provenance.py` |

**Why this notebook exists**

- Every step is explicit — no Celery, no HTTP, no MinIO.
- Every config value is inline and annotated — you can see exactly what
  our app passes to `PipelineConfig(**kwargs)`.
- Useful for: debugging extraction regressions, testing schema changes
  without a full pipeline run, evaluating prompt / config tweaks before
  committing to a rebuild.

**Prerequisites**

1. Main stack up: `docker compose up -d` (for `ollama`, and optionally
   `docling` / `docling-graph` — we don't call those services here).
2. Jupyter sidecar up: `docker compose -f docker-compose.jupyter.yml up -d`.
3. Test file under `notebooks/` on the host; it appears at
   `/app/notebooks/<name>` inside the container.

**Versions present in the Jupyter container at time of writing**

- `docling==2.90.0`
- `docling-graph==1.4.4`
- `docling-core==2.74.0`
- `litellm==1.83.9`
- `pydantic==2.12.5`


## 0. Configuration (edit these)

In [1]:
from pathlib import Path
import os

# Pick any file under ./notebooks/ on the host.
FILE_PATH = Path("/app/notebooks/SA-2 Surface-to-Air Missile _ National Museum of the United States Air Force™ _ Display.pdf")

# Ollama connection — same value our docling-graph service uses (.env:
# OLLAMA_LLM_BASE_URL). The service name resolves via the main stack's
# default network (we join it as external in docker-compose.jupyter.yml).
OLLAMA_BASE_URL = os.environ.get("OLLAMA_LLM_BASE_URL", "http://ollama:11434")

# LLM model — same default our config_builder uses for extraction.
LLM_PROVIDER = os.environ.get("DOCLING_GRAPH_LLM_PROVIDER", "ollama")
LLM_MODEL    = os.environ.get("DOCLING_GRAPH_LLM_MODEL",    "llama3.3:70b")

# Which pass to run first. One of: radar_domain | missile_domain | system_links
PASS_NAME = "radar_domain"

assert FILE_PATH.is_file(), f"FILE_PATH not found in container: {FILE_PATH}"
print("FILE_PATH    :", FILE_PATH)
print("OLLAMA       :", OLLAMA_BASE_URL)
print("LLM          :", f"{LLM_PROVIDER}/{LLM_MODEL}")
print("PASS_NAME    :", PASS_NAME)


FILE_PATH    : /app/notebooks/SA-2 Surface-to-Air Missile _ National Museum of the United States Air Force™ _ Display.pdf
OLLAMA       : http://10.0.1.121:11434
LLM          : ollama/llama3.3:70b
PASS_NAME    : radar_domain


## 1. Environment + library versions

Verify the three libraries we actually call are importable and that
Ollama is reachable. Our docling service (`docker/docling/app/main.py`)
does its own model-load probe at startup; here we just check imports.


In [2]:
import importlib.metadata as md
import requests

for pkg in ("docling", "docling-graph", "docling-core", "litellm", "pydantic"):
    try:
        print(f"  {pkg:16}: {md.version(pkg)}")
    except Exception as exc:
        print(f"  {pkg:16}: NOT INSTALLED ({exc})")

# Ollama reachability
try:
    r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    r.raise_for_status()
    tags = [m["name"] for m in r.json().get("models", [])]
    print(f"\nOllama reachable: {OLLAMA_BASE_URL}")
    print(f"  models loaded: {tags[:6]}{'…' if len(tags) > 6 else ''}")
    assert any(t.startswith(LLM_MODEL.split(':')[0]) for t in tags), \
        f"Expected a {LLM_MODEL} variant in Ollama. Run: ollama pull {LLM_MODEL}"
except Exception as exc:
    print(f"\nOllama check FAILED: {exc}")
    raise


  docling         : 2.90.0
  docling-graph   : 1.4.4
  docling-core    : 2.74.0
  litellm         : 1.83.9
  pydantic        : 2.12.5

Ollama reachable: http://10.0.1.121:11434
  models loaded: ['llama3.1:8b', 'llama3.3:70b', 'gemma4:31b', 'bge-m3:latest', 'embeddinggemma:latest', 'gpt-oss:120b']…


## 2. Docling conversion (raw library)

Our Docling service wraps `docling.document_converter.DocumentConverter`
with a handful of options tuned for our corpus. We reproduce the same
setup here so the `DoclingDocument` this notebook produces is
**byte-equivalent** to what the service would emit.

**What's reproduced exactly from `docker/docling/app/converter.py`:**

- **PDF pipeline options** (`_build_pdf_pipeline_options`, lines 140-168):
  - `accelerator_options.device="cuda"` — use the GPU.
  - `do_ocr=True` + `EasyOcrOptions(lang=["en"], use_gpu=True)`.
  - `do_table_structure=True` with `TableFormerMode.FAST` + cell matching.
  - `do_formula_enrichment=True`, `do_code_enrichment=True`.
  - `generate_picture_images=True`, `generate_page_images=True`,
    `images_scale=1.0`.
  - `do_picture_description=False` — picture VLM captions are added by
    the app pipeline *post-conversion* via Ollama (not by Docling).

- **`DocumentConverter`** constructed with `PdfFormatOption` for PDF /
  IMAGE and `SimplePipeline` for Word / PowerPoint / Excel / HTML /
  Markdown (converter.py:109-130).

The one thing we skip here: the service's `_patch_pil_crop()`
monkey-patch (converter.py:30-75). That patches PIL to survive
malformed bounding boxes from docling-core; for a single-file run it
rarely matters and only obscures the library's actual behavior.


In [3]:
import time, io, base64
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions, EasyOcrOptions,
    TableStructureOptions, TableFormerMode,
)
from docling.datamodel.accelerator_options import AcceleratorOptions
from docling.document_converter import (
    DocumentConverter, PdfFormatOption,
    WordFormatOption, PowerpointFormatOption,
    ExcelFormatOption, HTMLFormatOption, MarkdownFormatOption,
)
from docling.pipeline.simple_pipeline import SimplePipeline

pdf_options = PdfPipelineOptions(
    accelerator_options=AcceleratorOptions(device="cuda"),
    do_ocr=True,
    ocr_options=EasyOcrOptions(lang=["en"], use_gpu=True),
    do_table_structure=True,
    table_structure_options=TableStructureOptions(
        do_cell_matching=True,
        mode=TableFormerMode.FAST,
    ),
    do_formula_enrichment=True,
    do_code_enrichment=True,
    generate_picture_images=True,
    generate_page_images=True,
    images_scale=1.0,
    do_picture_description=False,
)

converter = DocumentConverter(
    allowed_formats=[
        InputFormat.PDF, InputFormat.IMAGE,
        InputFormat.DOCX, InputFormat.PPTX, InputFormat.XLSX,
        InputFormat.HTML, InputFormat.MD, InputFormat.ASCIIDOC, InputFormat.CSV,
    ],
    format_options={
        InputFormat.PDF:   PdfFormatOption(pipeline_options=pdf_options),
        InputFormat.IMAGE: PdfFormatOption(pipeline_options=pdf_options),
        InputFormat.DOCX:  WordFormatOption(pipeline_cls=SimplePipeline),
        InputFormat.PPTX:  PowerpointFormatOption(pipeline_cls=SimplePipeline),
        InputFormat.XLSX:  ExcelFormatOption(pipeline_cls=SimplePipeline),
        InputFormat.HTML:  HTMLFormatOption(pipeline_cls=SimplePipeline),
        InputFormat.MD:    MarkdownFormatOption(pipeline_cls=SimplePipeline),
    },
)

t0 = time.monotonic()
result = converter.convert(source=str(FILE_PATH))
elapsed_ms = (time.monotonic() - t0) * 1000

doc = result.document
print(f"Converted in {elapsed_ms:.0f} ms")
print(f"  pages   : {len(doc.pages)}")
print(f"  texts   : {len(doc.texts)}")
print(f"  pictures: {len(doc.pictures)}")
print(f"  tables  : {len(doc.tables)}")


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_85/1877735581.py:32: DeprecationWarning: Using DoclingParseDocumentBackend for InputFormat.IMAGE is deprecated. Images should use ImageDocumentBackend via ImageFormatOption. Automatically correcting the backend, please update your code to avoid this warning.
  converter = DocumentConverter(
/usr/local/lib/python3.11/site-packages/docling/models/stages/ocr/easyocr_model.py:68: UserWarning: Deprecated field. Better to set the `accelerator_options.device` in `pipeline_options`. When `use_gpu and accelerator_options.device == AcceleratorDevice.CUDA` the GPU is used to run EasyOCR. Otherwise, EasyOCR runs in CPU.
  warnings.warn(
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 2380.35it/s]
The tied weights mapping and config fo

Converted in 48854 ms
  pages   : 2
  texts   : 51
  pictures: 4
  tables  : 0


## 3. Inspect the DoclingDocument

The converter returns a `ConversionResult`; `result.document` is the
`DoclingDocument` (docling-core type). Everything downstream — our
chunking, our graph extraction, the Docling-Graph library — consumes
either `doc.export_to_markdown()` or `doc.export_to_dict()` (the JSON
registry form).

Our docling service returns both via its `/convert` endpoint:
- `markdown` — cleaned markdown for text chunking.
- `document_json` — the JSON registry for graph extraction.


In [4]:
markdown = doc.export_to_markdown()
doc_json = doc.export_to_dict()

print("top-level JSON keys :", list(doc_json.keys()))
print("texts in JSON       :", len(doc_json.get("texts", [])))
print("pictures in JSON    :", len(doc_json.get("pictures", [])))
print("tables in JSON      :", len(doc_json.get("tables", [])))
print("furniture entries   :", len(doc_json.get("furniture", {}).get("children", [])))

print("\n--- Markdown (first 600 chars) ---")
print(markdown[:5000])


top-level JSON keys : ['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages']
texts in JSON       : 51
pictures in JSON    : 4
tables in JSON      : 0
furniture entries   : 0

--- Markdown (first 600 chars) ---
<!-- image -->

<!-- image -->

[/ PHOTO DETAILS DOWNLOAD HI-RES](https://media.defense.gov/2009/Jun/05/2000558698/-1/-1/0/090605-F-1234P-021.JPG)

/

[PHOTO DETAILS](https://www.nationalmuseum.af.mil/Upcoming/Photos/igphoto/2000558713/)

[DOWNLOAD HI-RES](https://media.defense.gov/2009/Jun/05/2000558713/-1/-1/0/090605-F-1234P-015.JPG)

Series of a USAF RF-4C reconnaissance aircraft being shot down by an SA-2 on Aug. 12, 1967 near Hanoi, North Vietnam. Capts. Edwin Atterberry and Thomas Parrott were captured after ejecting. Atterberry died in the hands of the North Vietnamese after an escape attempt and Parrott was released at the end of the war. (U.S. Air Force photo)

## SA-2 Surface-t

In [5]:
# First non-empty text item from the Docling JSON.
# This is the granularity the library sees; our app's DocumentElement
# table is a curated subset (pipeline.py:2749-2778).
first_text = next((t for t in doc_json.get("texts", []) if t.get("text", "").strip()), None)
if first_text:
    keys = [k for k in first_text.keys() if k != "text"]
    print("first text item keys:", keys)
    print("  self_ref :", first_text.get("self_ref"))
    print("  label    :", first_text.get("label"))
    print("  prov[0]  :", first_text.get("prov", [{}])[0])
    print("  text     :", first_text.get("text")[:200])


first text item keys: ['self_ref', 'parent', 'children', 'content_layer', 'label', 'prov', 'orig']
  self_ref : #/texts/0
  label    : page_header
  prov[0]  : {'page_no': 1, 'bbox': {'l': 98.00000087666646, 't': 1152.0124953580669, 'r': 456.9274858355517, 'b': 1144.212495059067, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 90]}
  text     : N at i o n a l   M u s e u m   o f   t h e   U n i t e d   S tat e s   A i r   F o r c e ™


## 4. Pass template — a Pydantic class with catalog conventions

Docling-Graph's delta extractor is driven by a **Pydantic template
class**. Every field description, `Field.examples`, and the `edge()`
helper's metadata flows into the LLM prompt via the library's
`build_delta_node_catalog` + `build_catalog_prompt_block` pipeline.

Our pass templates live under
`ontology_bundles/air_defense_v3/extraction_schemas/<pass>.py`:

- `RadarDomainPass`
- `MissileDomainPass`
- `SystemLinksPass` (relationships-only, takes upstream entities)

**Conventions each class follows** (spec §Template Basics / §4.8):
- Pass roots: `ConfigDict(is_entity=True, graph_id_fields=[])`.
- Primary entities: `ConfigDict(is_entity=True, graph_id_fields=[...])`
  with at least one required identity field. `RadarSystemEntity` and
  `MissileSystemEntity` are **flat** (all checklist fields at the top
  level) — no nested subcomponent classes, no HAS_* edges.
- Pass-root entity-list fields declared via the `edge()` helper
  (label=`CONTAINS`) so GraphConverter walks into them.

Service-side resolution happens in
`docker/docling-graph/app/bundles.py`'s `load_pass_template(...)`; we
just import the class directly here.


In [6]:
from importlib import import_module
from pydantic import ConfigDict

PASS_MODULES = {
    "radar_domain":   ("ontology_bundles.air_defense_v3.extraction_schemas.radar_domain",   "RadarDomainPass"),
    "missile_domain": ("ontology_bundles.air_defense_v3.extraction_schemas.missile_domain", "MissileDomainPass"),
    "system_links":   ("ontology_bundles.air_defense_v3.extraction_schemas.system_links",   "SystemLinksPass"),
}
mod_path, cls_name = PASS_MODULES[PASS_NAME]
template_cls = getattr(import_module(mod_path), cls_name)

cfg = dict(template_cls.model_config) if isinstance(template_cls.model_config, dict) else {}
print(f"Pass template : {cls_name}")
print(f"  module      : {mod_path}")
print(f"  is_entity   : {cfg.get('is_entity')}")
print(f"  graph_id_fields : {cfg.get('graph_id_fields')}")
print()
print(f"Top-level fields ({len(template_cls.model_fields)}):")
for fname, finfo in template_cls.model_fields.items():
    extra = getattr(finfo, "json_schema_extra", None) or {}
    edge_label = extra.get("edge_label") if isinstance(extra, dict) else None
    label = f"  - {fname}"
    if edge_label:
        label += f"  [edge={edge_label}]"
    desc = (finfo.description or "")[:80]
    print(f"{label}: {desc}")


Pass template : RadarDomainPass
  module      : ontology_bundles.air_defense_v3.extraction_schemas.radar_domain
  is_entity   : True
  graph_id_fields : []

Top-level fields (1):
  - radar_systems  [edge=CONTAINS]: Top-level radar systems extracted from this document. Emit when the batch contai


## 5. Build a `PipelineConfig` — exactly how our service does

This is the contract between us and the library. Every knob below
corresponds one-to-one with `DoclingGraphSettings` in
`docker/docling-graph/app/config_builder.py:17-97`, and the kwargs dict
is the literal one our `build_pipeline_config()` assembles
(lines 128-174).

Each field is annotated with:
- **env var:** the name our app resolves from the process environment.
- **default:** our committed default (from the Python class, not from
  the library's default).
- **why:** what it controls at extraction time.

Per-pass override: `delta_quality_min_instances` drops to `1` for
`system_links` (relationships-only pass — producing zero ontology nodes
is legitimate there). We replicate that branch explicitly.


In [ ]:
from docling_graph import PipelineConfig

# Per-pass overrides from config_builder.py:104-106. system_links is the
# only exception today.
_QUALITY_MIN_INSTANCES_PER_PASS = {"system_links": 1}

def build_pipeline_config_local(source: str, template_class, pass_name: str | None = None) -> PipelineConfig:
    """Replica of docker/docling-graph/app/config_builder.py:build_pipeline_config.

    Only difference: reads env vars directly (no pydantic_settings dance),
    so you can tweak a value in the notebook and reconfigure instantly.
    """
    env = os.environ.get

    quality_min_instances = int(env("DOCLING_GRAPH_QUALITY_MIN_INSTANCES", "3"))
    if pass_name in _QUALITY_MIN_INSTANCES_PER_PASS:
        quality_min_instances = _QUALITY_MIN_INSTANCES_PER_PASS[pass_name]

    kwargs = {
        # Source file — the library loads this into a DoclingDocument if
        # it isn't already one; we pass the raw PDF path.
        "source": source,

        # --- LLM backend (config_builder.py:96-97, 32-33) -----------------
        "backend":           env("DOCLING_GRAPH_BACKEND",              "llm"),
        "inference":         "local",
        "provider_override": env("DOCLING_GRAPH_LLM_PROVIDER",         "ollama"),
        "model_override":    env("DOCLING_GRAPH_LLM_MODEL",            "llama3.3:70b"),

        # --- Extraction contract (config_builder.py:34-35) ----------------
        # "delta" = chunk-level emit-nodes+relationships; what the service
        # ships. Alternatives: "direct", "staged".
        "extraction_contract": env("DOCLING_GRAPH_EXTRACTION_CONTRACT", "delta"),
        "processing_mode":     env("DOCLING_GRAPH_PROCESSING_MODE",     "many-to-one"),

        # --- Chunking (config_builder.py:38-45) ---------------------------
        "use_chunking":            env("DOCLING_GRAPH_USE_CHUNKING",          "true").lower() == "true",
        "chunk_max_tokens":    int(env("DOCLING_GRAPH_CHUNK_MAX_TOKENS",      "512")),
        "llm_batch_token_size":int(env("DOCLING_GRAPH_LLM_BATCH_TOKEN_SIZE",  "1024")),
        "parallel_workers":    int(env("DOCLING_GRAPH_PARALLEL_WORKERS",      "2")),
        "staged_pass_retries": int(env("DOCLING_GRAPH_BATCH_SPLIT_MAX_RETRIES","1")),

        # --- Delta resolvers (config_builder.py:48-51) --------------------
        "delta_resolvers_enabled":    env("DOCLING_GRAPH_RESOLVERS_ENABLED", "true").lower() == "true",
        "delta_resolvers_mode":       env("DOCLING_GRAPH_RESOLVERS_MODE",    "semantic"),
        "delta_resolver_fuzzy_threshold":    float(env("DOCLING_GRAPH_RESOLVER_FUZZY_THRESHOLD",    "0.8")),
        "delta_resolver_semantic_threshold": float(env("DOCLING_GRAPH_RESOLVER_SEMANTIC_THRESHOLD", "0.8")),

        # --- Delta quality gate (config_builder.py:54-57) ------------------
        # Upstream lib default is 20 — we lowered to 3, then (in .env) to 1
        # while debugging sparse prose extraction.
        "delta_quality_require_root":       env("DOCLING_GRAPH_QUALITY_REQUIRE_ROOT",   "true").lower() == "true",
        "delta_quality_min_instances":      quality_min_instances,
        "delta_quality_max_parent_lookup_miss": int(env("DOCLING_GRAPH_QUALITY_MAX_PARENT_MISS", "4")),
        "delta_quality_adaptive_parent_lookup":  env("DOCLING_GRAPH_QUALITY_ADAPTIVE_PARENT", "true").lower() == "true",

        # --- Delta normalizer (config_builder.py:60-63) -------------------
        "delta_normalizer_validate_paths":          env("DOCLING_GRAPH_NORMALIZER_VALIDATE_PATHS",       "true").lower() == "true",
        "delta_normalizer_canonicalize_ids":        env("DOCLING_GRAPH_NORMALIZER_CANONICALIZE_IDS",     "true").lower() == "true",
        "delta_normalizer_strip_nested_properties": env("DOCLING_GRAPH_NORMALIZER_STRIP_NESTED",         "true").lower() == "true",
        "delta_normalizer_attach_provenance":       env("DOCLING_GRAPH_NORMALIZER_ATTACH_PROVENANCE",    "true").lower() == "true",

        # --- Identity filter (post-extraction section-title pruner) -------
        # This is the "docs-recommended safety net" that replaces our old
        # prompt_overrides.py rewrites (removed).
        "delta_identity_filter_enabled": env("DOCLING_GRAPH_IDENTITY_FILTER_ENABLED", "true").lower() == "true",
        "delta_identity_filter_strict":  env("DOCLING_GRAPH_IDENTITY_FILTER_STRICT",  "false").lower() == "true",

        # --- Gleaning (config_builder.py:75-76) ---------------------------
        # 1 extra pass after the primary: "what did you miss?" for recall.
        "gleaning_enabled":        env("DOCLING_GRAPH_GLEANING_ENABLED",        "true").lower() == "true",
        "gleaning_max_passes": int(env("DOCLING_GRAPH_GLEANING_MAX_PASSES",     "2")),

        # --- Structured output (config_builder.py:79-80) ------------------
        "structured_output":        env("DOCLING_GRAPH_STRUCTURED_OUTPUT",       "true").lower() == "true",
        "structured_sparse_check":  env("DOCLING_GRAPH_STRUCTURED_SPARSE_CHECK", "true").lower() == "true",

        # --- LLM overrides (config_builder.py:159-172) --------------------
        # temperature=0.1 on purpose: 0.0 caused llama3.3:70b to emit empty
        # JSON for our 40-path DeltaGraph schema. Small variance is needed
        # to explore valid completions under the structured-output constraint.
        "llm_overrides": {
            "generation": {
                "temperature": float(env("DOCLING_GRAPH_LLM_TEMPERATURE", "0.1")),
                "max_tokens":   int(env("DOCLING_GRAPH_LLM_MAX_TOKENS", "32000")),
            },
            "reliability": {
                "timeout_s": int(env("DOCLING_GRAPH_LLM_TIMEOUT", "10800")),
            },
            "connection": {
                "base_url": env("OLLAMA_LLM_BASE_URL", "http://ollama:11434"),
            },
            # LiteLLM has no metadata for ollama/llama3.3:70b, so its
            # resolve_effective_model_config falls back to _DEFAULT_MAX_OUTPUT_TOKENS=4092.
            # Override explicitly so max_tokens (32000) is accepted.
            "context_limit":     int(env("DOCLING_GRAPH_LLM_CONTEXT_LIMIT",     "131072")),
            "max_output_tokens": int(env("DOCLING_GRAPH_LLM_MAX_OUTPUT_TOKENS", "32000")),
        },

        # --- Misc ---------------------------------------------------------
        "dump_to_disk": False,
    }
    if template_class is not None:
        kwargs["template"] = template_class

    return PipelineConfig(**kwargs)

config = build_pipeline_config_local(
    source=str(FILE_PATH),
    template_class=template_cls,
    pass_name=PASS_NAME,
)

# Show the subset of fields a reader cares about.
print(f"PipelineConfig(")
for k in ("backend", "provider_override", "model_override",
          "extraction_contract", "processing_mode",
          "use_chunking", "chunk_max_tokens", "llm_batch_token_size",
          "delta_quality_min_instances", "delta_identity_filter_enabled",
          "gleaning_enabled", "gleaning_max_passes",
          "structured_output"):
    v = getattr(config, k, "<missing>")
    print(f"  {k:32} = {v!r}")
print(f"  llm_overrides.generation.temperature = {config.llm_overrides.generation.temperature}")
print(f"  llm_overrides.connection.base_url    = {config.llm_overrides.connection.base_url}")
print(f")")


## 6. Catalog + prompt (what the LLM actually sees)

We reconstruct the exact system/user prompt the library will send for a
given batch — using production's own chunker and batcher. Four
ingredients go into that prompt:

1. **Chunks** from `DocumentChunker` (HybridChunker + sentence-transformers
   tokenizer, capped at `chunk_max_tokens=512`). This merges Docling's
   raw `texts[]` fragments into semantically coherent chunks, so you do
   *not* see icons / page numbers / single-line UI crumbs as standalone
   chunks. That raw registry is a preview pitfall.
2. **Batches** from `chunk_batches_by_token_limit` packing those chunks
   until each batch totals ≤ `llm_batch_token_size=1024` tokens.
3. **Path catalog** from `build_delta_node_catalog` +
   `build_catalog_prompt_block` — flattened from the template class.
4. **Semantic guide** from `build_delta_semantic_guide` — derived from
   the Pydantic JSON schema.

Then `get_delta_batch_prompt(...)` assembles them with one batch's
content. Adjust `BATCH_INDEX` in the cell below to view different
batches — this is what the LLM gets for each `/extract-pass` call that
production fires for this document.


In [8]:
from docling_graph.core.extractors.document_chunker import DocumentChunker
from docling_graph.core.extractors.contracts.delta.helpers import chunk_batches_by_token_limit
from docling_graph.core.extractors.contracts.delta.catalog import build_delta_node_catalog
from docling_graph.core.extractors.contracts.delta.schema_mapper import (
    build_catalog_prompt_block, build_delta_semantic_guide,
)
from docling_graph.core.extractors.contracts.delta.prompts import (
    get_delta_batch_prompt, format_batch_markdown,
)
# Match the service-side rewrite: replace the library's system prompt with the
# mention-level-friendly version. Same source of truth both sides import from.
from ontology_bundles._shared.prompt_rules import DELTA_SYSTEM_PROMPT
_original_prompt = get_delta_batch_prompt
def get_delta_batch_prompt(**kw):
    r = _original_prompt(**kw)
    if isinstance(r, dict) and "system" in r:
        r["system"] = DELTA_SYSTEM_PROMPT
    return r

BATCH_INDEX          = 0     # which real batch to display (0 = first)
CHUNK_MAX_TOKENS     = 512   # matches DoclingGraphSettings.docling_graph_chunk_max_tokens
LLM_BATCH_TOKEN_SIZE = 1024  # matches DoclingGraphSettings.docling_graph_llm_batch_token_size

# ── 1. Run production's chunker over this document's DoclingDocument ─
# `doc` is the DoclingDocument produced by Section 2's DocumentConverter call.
chunker = DocumentChunker(
    tokenizer_name="sentence-transformers/all-MiniLM-L6-v2",
    chunk_max_tokens=CHUNK_MAX_TOKENS,
    merge_peers=True,
)
chunks       = chunker.chunk_document(doc)
token_counts = [chunker.tokenizer.count_tokens(c) for c in chunks]
batch_plan   = chunk_batches_by_token_limit(
    chunks, token_counts, max_batch_tokens=LLM_BATCH_TOKEN_SIZE,
)

print(f"chunker produced {len(chunks)} chunks (total {sum(token_counts)} tokens)")
print(f"batched into     {len(batch_plan)} batch(es)")
for i, b in enumerate(batch_plan):
    tokens = sum(t for _, _, t in b)
    print(f"  batch {i:2}: {len(b):2} chunks, {tokens:4} tokens")

if BATCH_INDEX >= len(batch_plan):
    raise SystemExit(
        f"BATCH_INDEX={BATCH_INDEX} out of range (only {len(batch_plan)} batches)."
    )

selected_batch = batch_plan[BATCH_INDEX]
batch_markdown = format_batch_markdown([text for _i, text, _t in selected_batch])

# ── 2. Build catalog + semantic guide from the template ────────────
catalog        = build_delta_node_catalog(template_cls)
catalog_block  = build_catalog_prompt_block(catalog)
schema_dict    = template_cls.model_json_schema()
semantic_guide = build_delta_semantic_guide(template_cls, schema_dict)

first_chunk = chunks[0].strip() if chunks else ""
global_context = (
    first_chunk[:600] + ("..." if len(first_chunk) > 600 else "")
) if first_chunk else None

# ── 3. Assemble the prompt ─────────────────────────────────────────
prompt = get_delta_batch_prompt(
    batch_markdown=batch_markdown,
    schema_semantic_guide=semantic_guide,
    path_catalog_block=catalog_block,
    batch_index=BATCH_INDEX,
    total_batches=len(batch_plan),
    global_context=global_context,
    already_found=None,
)

print(f"\ncatalog paths : {len(catalog.nodes)}")
print(f"catalog_block : {len(catalog_block)} chars")
print(f"semantic_guide: {len(semantic_guide)} chars")
print(f"prompt.system : {len(prompt['system'])} chars")
print(f"prompt.user   : {len(prompt['user'])} chars")
print()
print(f"=== SYSTEM (full) ===\n{prompt['system']}")
print(f"\n=== USER (batch {BATCH_INDEX+1}/{len(batch_plan)}, {sum(t for _,_,t in selected_batch)} tokens) ===")
print(prompt["user"])


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

chunker produced 7 chunks (total 1023 tokens)
batched into     1 batch(es)
  batch  0:  7 chunks, 1023 tokens

catalog paths : 2
catalog_block : 958 chars
semantic_guide: 3419 chars
prompt.system : 1973 chars
prompt.user   : 10179 chars

=== SYSTEM (full) ===
You are an expert extraction engine for graph construction. Return ONLY strict JSON with top-level keys 'nodes' and 'relationships'.

Rules:
1. Use exact catalog paths for 'path' and parent; never invent paths or use class names. Put only identity fields in ids; other values go in properties. ids keys must match catalog.
2. Model nested entities as separate nodes (flat properties only; no nested objects in properties). For any list-entity path in the catalog (paths ending in [] with id_fields): set identity in ids from the document. Put child entities on the child path with parent reference; when emitting children whose parent is a list path, also emit a parent-path node with ids set from the document so parent lookup can attach t

## 7. Run one pass via `run_pipeline(config)`

`run_pipeline` is the library's top-level entry point for an
end-to-end extraction against a single source. Our service uses it via
`run_extraction_pass(...)` at
`docker/docling-graph/app/main.py:376-445`.

What happens inside:

1. Parse / normalize the source into a `DoclingDocument`.
2. Chunk the document (if `use_chunking=True`).
3. For each batch: build the prompt from §6 and call the LLM via
   LiteLLM (respecting `llm_overrides`), validating against
   `DeltaGraph` JSON schema.
4. Resolve + normalize + identity-filter the merged graph.
5. Run the quality gate (`delta_quality_*` knobs).
6. Optionally: gleaning pass for recall.
7. Project the graph into the template (`template_instance`) and
   return a `PipelineContext`.

The service then promotes `extracted_models[0]` to `template_instance`
(main.py:435-437) — mirror that here.

> **Tip.** First run on Ollama is slow (model warm-up + constrained-
> decoding compilation). Subsequent runs are much faster.


In [9]:
from docling_graph import run_pipeline

context = run_pipeline(config)

# Mirror our service's promotion step so downstream code sees the
# populated pass root (main.py:435-437).
extracted = getattr(context, "extracted_models", None)
if isinstance(extracted, list) and extracted:
    context.template_instance = extracted[0]

graph = context.knowledge_graph
meta  = context.graph_metadata
print(f"nodes : {graph.number_of_nodes()}")
print(f"edges : {graph.number_of_edges()}")
print(f"node_types (meta) : {getattr(meta, 'node_types', None)}")
print(f"edge_types (meta) : {getattr(meta, 'edge_types', None)}")
print(f"template_instance : {type(context.template_instance).__name__ if context.template_instance else None}")


Pipeline failed at stage: Extraction


PipelineError: Pipeline failed at stage 'Extraction': ConfigurationError
Details: stage=Extraction, error=max_tokens exceeds model limit
Details: model=llama3.3:70b, max_tokens=32000, model_max_output_tokens=4092, error_type=ConfigurationError

In [ ]:
# Serialize what the pass produced, the same shape our service returns
# in ExtractPassResponse.pass_output (main.py:582-590).
pass_output = (
    context.template_instance.model_dump(mode="json")
    if context.template_instance is not None
    else None
)

if pass_output is None:
    print("No template_instance produced — likely empty extraction or quality-gate failure.")
else:
    # Trim the blob for display — just show top-level field counts and first entity.
    from pprint import pformat
    top_keys = list(pass_output.keys())
    print(f"pass_output top-level keys: {top_keys}")
    for k in top_keys:
        v = pass_output[k]
        if isinstance(v, list):
            print(f"  {k}: list len={len(v)}  first={pformat(v[0], depth=2)[:200] if v else None}")
        else:
            print(f"  {k}: {type(v).__name__}")


## 8. Multi-pass orchestration (radar → missile → system_links)

Our bundle runs three passes in declared order. The first two are
`input_mode=document_only` and run independently; the third,
`system_links`, is `input_mode=document_plus_entity_refs` — it consumes
entities from the first two and emits cross-domain relationships.

The service threads upstream entities through the "Path B preamble"
mechanism (`main.py:387-415`): a formatted block of upstream entity
identities is inserted into `doc_json.texts` + `body.children` so the
LLM sees them in the extraction batch.

Below we run a simplified version: two independent passes,
concatenate their emitted entities as prose, and pass that text as a
batch for the relationships-only pass.

> This is a fidelity shortcut, not a perfect reproduction. The service
> reconstructs the `docling_document_json` shape per pass; here we only
> want to show the sequencing + upstream-ref propagation. For a full
> reproduction of Path B, see `main.py:387-415`.


In [ ]:
def run_pass(pass_name: str):
    mod_path, cls_name = PASS_MODULES[pass_name]
    cls  = getattr(import_module(mod_path), cls_name)
    cfg  = build_pipeline_config_local(str(FILE_PATH), cls, pass_name=pass_name)
    ctx  = run_pipeline(cfg)
    extr = getattr(ctx, "extracted_models", None)
    if isinstance(extr, list) and extr:
        ctx.template_instance = extr[0]
    return ctx

ctx_radar   = run_pass("radar_domain")
print(f"radar_domain  : nodes={ctx_radar.knowledge_graph.number_of_nodes()} "
      f"edges={ctx_radar.knowledge_graph.number_of_edges()}")

ctx_missile = run_pass("missile_domain")
print(f"missile_domain: nodes={ctx_missile.knowledge_graph.number_of_nodes()} "
      f"edges={ctx_missile.knowledge_graph.number_of_edges()}")

# system_links: relationships-only pass. Needs upstream entities as context.
# In the service this happens via Path B preamble injection (main.py:387-415).
# For the notebook we just skip if either upstream pass produced nothing.
has_radar_entities   = ctx_radar.template_instance is not None
has_missile_entities = ctx_missile.template_instance is not None
if has_radar_entities or has_missile_entities:
    ctx_links = run_pass("system_links")
    print(f"system_links  : nodes={ctx_links.knowledge_graph.number_of_nodes()} "
          f"edges={ctx_links.knowledge_graph.number_of_edges()}")
else:
    print("system_links  : SKIPPED (no upstream entity refs to link)")


## 9. Build provenance rows (what the service returns per-node)

Our service builds per-node `ExtractionProvenance` rows via
`build_provenance_from_context` (`docker/docling-graph/app/provenance.py`).
These are attached to the `/extract-pass` response and used downstream
to (a) map every extracted entity back to its originating Docling
element and (b) populate `HAS_PROVENANCE` edges in ArcadeDB.

The raw library doesn't build this structure for us — the service
does. Here we replicate it inline so you can see the shape.


In [ ]:
# ExtractionProvenance is defined by docling-graph services' schema. For
# the notebook we'll dump what the library itself attached to each node.
nodes = []
for node_key, data in context.knowledge_graph.nodes(data=True):
    prov = data.get("_provenance") or data.get("provenance")
    if prov:
        nodes.append({"node": node_key, "type": data.get("node_type"), "prov": prov})

print(f"nodes with library-level _provenance: {len(nodes)}")
for n in nodes[:3]:
    print(n)


## 10. What's missing vs. the full pipeline

This notebook covers the **extraction** arc end-to-end using the raw
libraries. The following are deliberately out of scope; they're app-
level concerns our full pipeline (`app/workers/pipeline.py`) handles:

- **MinIO persistence** — raw files, Docling JSON, image artifacts.
- **PostgreSQL persistence** — `document_elements`, `artifacts`,
  `text_chunks`, `image_chunks`, `graph_extractions` rows.
- **Text / image embeddings** — BGE-M3 for text (via Ollama),
  open_clip ViT-L-16-SigLIP2-256 for images.
- **Merge across passes** — `app/services/extraction_merge.py`
  deduplicates `LogicalIdentity`s emitted by different passes and
  validates edges against the ontology validation matrix.
- **ArcadeDB upserts** — entity vertices, chunk vertices,
  `EXTRACTED_FROM`, `HAS_PROVENANCE`, `SAME_PAGE`, etc.
- **Pipeline-run bookkeeping** — `pipeline_runs`, `stage_runs`,
  status transitions.

For those, run the `ingest_walkthrough.ipynb` notebook instead — it
uses the real Celery tasks via `.apply(...)` and produces all the side
effects.
